## Настройки Colab

In [1]:
# Настройка пользователя (сделать один раз)
!git config --global user.email "nabludatellip@gmail.com"
!git config --global user.name "ProninPV"

In [2]:
from getpass import getpass

# 1. Безопасный ввод токена
GITHUB_TOKEN = getpass('Введите ваш GitHub Personal Access Token: ')

Введите ваш GitHub Personal Access Token: ··········


In [3]:
!git clone https://github.com/ProninPV/ml-regression_concrete-strength.git
%cd ml-regression_concrete-strength

Cloning into 'ml-regression_concrete-strength'...
remote: Enumerating objects: 530, done.
remote: Counting objects: 100% (530/530), done.
remote: Compressing objects: 100% (263/263), done.
remote: Total 530 (delta 282), reused 468 (delta 226), pack-reused 0 (from 0)
Receiving objects: 100% (530/530), 8.96 MiB | 13.02 MiB/s, done.
Resolving deltas: 100% (282/282), done.
/content/ml-regression_concrete-strength


In [4]:
%cd /content/ml-regression_concrete-strength
!git config pull.rebase false
!git pull origin submit_1

/content/ml-regression_concrete-strength
From https://github.com/ProninPV/ml-regression_concrete-strength
 * branch            submit_1   -> FETCH_HEAD
Updating 89655eb..9c9bf07
Fast-forward
 catboost_info/catboost_training.json               | 200 +++----
 catboost_info/learn/events.out.tfevents            | Bin 4798 -> 4798 bytes
 catboost_info/time_left.tsv                        | 200 +++----
 config/config.yaml                                 | 378 +++++++-------
 models/modeling_report/modeling_experiments.csv    | 220 ++++----
 .../modeling_experiments_20251115_135319.csv       |  11 -
 .../modeling_experiments_20251115_140340.csv       |  11 -
 .../modeling_experiments_20251115_140649.csv       |  11 -
 .../modeling_experiments_20251115_141542.csv       |  11 -
 .../modeling_experiments_20251115_141948.csv       |  11 -
 .../modeling_experiments_20251115_142253.csv       |  11 -
 .../modeling_experiments_20251115_142436.csv       |  11 -
 models/pipelines/pipeline_Ridge_6_5015.

In [7]:
!git branch

* submit_1
  tuning


In [8]:
!git checkout submit_1

Already on 'submit_1'
Your branch is up to date with 'origin/submit_1'.


In [9]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 8.4 MB/s eta 0:00:00


In [10]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 404.7/404.7 kB 6.2 MB/s eta 0:00:00


## 6.0 Импорты библиотек

In [2]:
import os
import yaml
import logging
import pickle
import numpy as np
import scipy.stats as stats
import sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import levene
from scipy.stats import ttest_ind
from typing import List, Any, Optional, Tuple, Dict, Union
from datetime import datetime
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import cross_val_score, KFold, RepeatedKFold
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import make_scorer, mean_squared_error
from sklearn.ensemble import RandomForestRegressor
from catboost import CatBoostRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.ensemble import AdaBoostRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import VotingRegressor, StackingRegressor
from sklearn.compose import ColumnTransformer
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
import time
import psutil
from tqdm import tqdm
import gc
import optuna
import joblib

D:\Skills\Kaggle\ml-regression_concrete-strength\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
!pip install -U lightautoml


KeyboardInterrupt



In [12]:
import warnings
warnings.filterwarnings("ignore")

In [13]:
# расширяем поле ноутбука для удобства
from IPython.display import display, HTML
display(HTML('<style>.container {width:87% !important;}</style>'))
display(HTML("<style>.output_scroll {height:auto !important; max-height:10000px !important;}</style>"))

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

In [14]:
# Настройки для pandas (количество отображаемых колонок)
pd.set_option('display.max_columns', 100)

In [15]:
# Определение стиля для pyplot
plt.style.use('ggplot')

In [16]:
# В Colab проект клонируется в /content/
# Устанавливаем правильную рабочую директорию
# project_root = Path('/content/ml-regression_concrete-strength')

# Определяем корень проекта
cwd = Path().resolve()
project_root = cwd.parent

# Добавляем корень проекта в sys.path (этого достаточно)
sys.path.append(str(project_root))

# Проверяем наличие конфиг файла
config_path = project_root / "config" / "config.yaml"
print(f"Looking for config at: {config_path}")

# Загрузка данных из config.yaml
from src.data import downloader, loader, preprocessor, saving
from src.features import feat_preprocessing
from src.modeling import modeling

# Передаем путь явно
config = loader.load_config(config_path)
print("✅ Config loaded successfully!")

Looking for config at: /content/ml-regression_concrete-strength/config/config.yaml
✅ Config loaded successfully!


## 6.1. Загрузка данных

In [22]:
# Загрузка train
df_train = loader.data_load_preprocessed(data_type='train',
                                         config=config)

[⧗] Загружаю данные из: /content/ml-regression_concrete-strength/data/processed/eda_data_train.pkl
[✓] Данные успешно загружены. Форма: (781, 11)


In [23]:
# Вывод первых 5 строк тренировочного датасета
df_train.head()

,Cement,Blast Furnace Slag,Fly Ash,Water,Superplasticizer,Coarse Aggregate,Fine Aggregate,Age,Strength,W/C,Sp/C_pct
0,376.0,0.0,0.0,214.6,0.0,1003.5,762.4,3,16.28,0.570745,0.000000
1,491.0,26.0,123.0,210.0,3.9,882.0,699.0,56,59.59,0.427699,0.007943
2,250.0,0.0,95.7,187.4,5.5,956.9,861.2,3,13.82,0.749600,0.022000
3,310.0,0.0,0.0,192.0,0.0,1012.0,830.0,90,35.76,0.619355,0.000000
4,252.1,97.1,75.6,193.8,8.3,835.5,821.4,28,33.40,0.768743,0.032923


## 6.2. Обучение LightAutoML

In [ ]:
task = Task('reg')

In [ ]:
target = 'Strength'
rs = 42

In [ ]:
roles = {
    'target': target
}

In [ ]:
timeout = 3600
threads = 4
cv = 5

In [ ]:
%%time
automl = TabularAutoML(task = task,
                       timeout = timeout,
                       cpu_limit = threads,
                       reader_params = {'n_jobs': threads, 'random_state': rs, 'cv': cv})


oof_pred = automl.fit_predict(tr_data, roles = roles, verbose = 3)

## Отправка на Github


In [44]:
!git status

On branch submit_1
Your branch is up to date with 'origin/submit_1'.

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   catboost_info/catboost_training.json
	modified:   catboost_info/learn/events.out.tfevents
	modified:   catboost_info/learn_error.tsv
	modified:   catboost_info/time_left.tsv

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	models/pipelines/pipeline_catboostregression_3_9566.pkl
	models/pipelines/pipeline_lightgbm_4_1400.pkl
	models/pipelines/pipeline_randomforest_4_8069.pkl
	models/tuning_models/

no changes added to commit (use "git add" and/or "git commit -a")


In [45]:
# 1. Добавляем файлы
!git add .
!git add models/tuning_models
!git add models/pipelines

# 2. Коммитим
!git commit -m "fit: add tuning XGboost regression"

[submit_1 eec7452] fit: add tuning XGboost regression
 10 files changed, 5586 insertions(+), 306 deletions(-)
 rewrite catboost_info/catboost_training.json (94%)
 rewrite catboost_info/learn/events.out.tfevents (93%)
 rewrite catboost_info/learn_error.tsv (98%)
 rewrite catboost_info/time_left.tsv (95%)
 create mode 100644 models/pipelines/pipeline_catboostregression_3_9566.pkl
 create mode 100644 models/pipelines/pipeline_lightgbm_4_1400.pkl
 create mode 100644 models/pipelines/pipeline_randomforest_4_8069.pkl
 create mode 100644 models/tuning_models/catboostregression_3_9566.pkl
 create mode 100644 models/tuning_models/lightgbm_4_1400.pkl
 create mode 100644 models/tuning_models/randomforest_4_8069.pkl


In [46]:
!git push https://{GITHUB_TOKEN}@github.com/ProninPV/ml-regression_concrete-strength.git submit_1

Enumerating objects: 26, done.
Counting objects: 100% (26/26), done.
Delta compression using up to 2 threads
Compressing objects: 100% (16/16), done.
Writing objects: 100% (17/17), 15.56 MiB | 3.22 MiB/s, done.
Total 17 (delta 4), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (4/4), completed with 1 local object.
remote: warning: See https://gh.io/lfs for more information.
remote: warning: File models/pipelines/pipeline_randomforest_4_8069.pkl is 63.05 MB; this is larger than GitHub's recommended maximum file size of 50.00 MB
remote: warning: File models/tuning_models/randomforest_4_8069.pkl is 62.89 MB; this is larger than GitHub's recommended maximum file size of 50.00 MB
remote: warning: GH001: Large files detected. You may want to try Git Large File Storage - https://git-lfs.github.com.
To https://github.com/ProninPV/ml-regression_concrete-strength.git
   9c9bf07..eec7452  submit_1 -> submit_1
